## Simple implemenation of Bifrost


**Installing Env Variables**

In [36]:
import os
from dotenv import load_dotenv

# Load secrets from .env — never hardcode API keys in this notebook.
# Re-run this cell after editing .env (no kernel restart needed).
load_dotenv(override=True)

REQUIRED_ENV_VARS = (
    "OPENAI_API_KEY",
    "GROQ_API_KEY",
    "BIFROST_OPENAI_API_KEY",
    "BIFROST_GROQ_API_KEY",
    "BIFROST_BASE_URL"
)

missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Copy .env.example to .env and set your values."
    )

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BIFROST_OPENAI_API_KEY = os.getenv("BIFROST_OPENAI_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_BASE_URL = os.getenv("BIFROST_BASE_URL")

print("Environment loaded.")


Environment loaded.


**Health Check**

In [37]:
import httpx
try:
    health = httpx.get(f"{BIFROST_BASE_URL}/health")
    print(health.json())
except Exception as e:
    print(f"Error: {e}")


{'components': {'db_pings': 'ok'}, 'status': 'ok'}


### Reusable Text Prompts and Utilits

In [38]:
### Reusable Text Prompts and Utilits
TEST_PROMPTS = {
    "simple": "What is the capital of France?",
    "reasoning": "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code": "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki": "What are the stream modes in the new langgraph version? Use the deepwiki",
    "tavily": "Search the web for the latest news about Groq AI and summarize the top"
}

import time
def time_caputer(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        return result, end_time - start_time
    return wrapper  # <-- this was missing


CHATOPENAI_MODEL = "openai/gpt-4o-mini"
CHATOPENAI_FALLBACK_MODEL = "openai/gpt-5.6-luna"

# provider/model format — groq/ routes to Groq, openai/ routes to OpenAI
GROQ_MODEL = "groq/openai/gpt-oss-20b"
GROQ_FALLBACK_MODEL = "openai/gpt-4o-mini"

MODEL_DEFAULT_FALLBACKS = {
    CHATOPENAI_MODEL: CHATOPENAI_FALLBACK_MODEL,
    GROQ_MODEL: GROQ_FALLBACK_MODEL,
}


**Bad Approach Calling direct model**

No caching , routing , virtual keys can be applied , for this we need to make extra efforts 

In [39]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses three primary types of telemetry data: traces, metrics, and logs, enabling developers to gain comprehensive insights into their systems' performance and behavior. By offering a unified approach, OpenTelemetry simplifies the instrumentation process, allowing developers to focus on building applications rather than worrying about the intricacies of monitoring.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, which means it can be integrated with various back-end systems and observability tools. This flexibility allows organizations to choose the best solutions for their needs without being locked into a specific vendor. Additionally, OpenTelemetry supports multiple programming languages, making it accessible for diverse development environments.

As cloud-native architectures and microserv

## Using the BiFrost LLM Gateway

In [40]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model=CHATOPENAI_MODEL,              # "openai/gpt-4o-mini"
    base_url=f"{BIFROST_BASE_URL}/langchain",
    api_key=BIFROST_OPENAI_API_KEY,      # not GROQ key
    temperature=0,
    streaming=True,
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses three primary types of telemetry: traces, metrics, and logs, enabling developers to gain comprehensive insights into their systems' performance and behavior. By unifying these data types, OpenTelemetry facilitates a holistic view of application health, making it easier to diagnose issues and optimize performance.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, allowing organizations to choose their preferred backend for data storage and analysis. This flexibility empowers teams to avoid vendor lock-in while still benefiting from powerful observability tools. OpenTelemetry supports a wide range of programming languages and frameworks, making it accessible for diverse technology stacks.

The framework promotes best practices in observability, encouraging developers to instrument their cod

**Bifrost Helper function**


In [41]:
import json as _json

from openai import OpenAI

DEFAULT_CACHE_KEY = "notebook-demo"

_bifrost_clients: dict[str, OpenAI] = {}


def _provider_from_model(model: str) -> str:
    return model.split("/", 1)[0] if "/" in model else "openai"


def _bifrost_client(model: str) -> OpenAI:
    provider = _provider_from_model(model)
    api_key = BIFROST_GROQ_API_KEY if provider == "groq" else BIFROST_OPENAI_API_KEY

    if provider not in _bifrost_clients:
        _bifrost_clients[provider] = OpenAI(
            base_url=f"{BIFROST_BASE_URL}/openai",
            api_key=api_key,
        )
    return _bifrost_clients[provider]


def _resolve_fallbacks(model: str, fallback_model: str | None, fallback_models: list[str] | None) -> list[str]:
    if fallback_models:
        return list(fallback_models)
    if fallback_model:
        return [fallback_model]
    return [MODEL_DEFAULT_FALLBACKS[model]] if model in MODEL_DEFAULT_FALLBACKS else []


def call_bifrost(
    prompt: str,
    model: str,
    fallback_model: str | None = None,
    fallback_models: list[str] | None = None,
    cache_key: str | None = DEFAULT_CACHE_KEY,
    cache_type: str | None = "direct",
):
    client = _bifrost_client(model)
    fallbacks = _resolve_fallbacks(model, fallback_model, fallback_models)

    headers = {}
    if cache_key:
        headers["x-bf-cache-key"] = cache_key
    if cache_type:
        headers["x-bf-cache-type"] = cache_type

    request_kwargs = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    if headers:
        request_kwargs["extra_headers"] = headers
    if fallbacks:
        # Bifrost reads fallbacks from the JSON body (not only from headers).
        request_kwargs["extra_body"] = {"fallbacks": fallbacks}

    raw_response = client.chat.completions.with_raw_response.create(**request_kwargs)

    response = raw_response.parse()
    body = _json.loads(raw_response.text)
    extra = body.get("extra_fields", {})
    cache_debug = extra.get("cache_debug", {})

    fallback_index = raw_response.headers.get("x-bifrost-fallback-index")
    print(
        "provider="
        f"{extra.get('provider')} "
        f"resolved_model={extra.get('resolved_model_used')} "
        f"fallback_index={fallback_index or '0 (primary)'} "
        f"cache_hit={cache_debug.get('cache_hit')} "
        f"hit_type={cache_debug.get('hit_type')}"
    )
    if fallbacks:
        print(f"fallbacks={fallbacks}")

    return response.choices[0].message.content, body


In [44]:
content1, meta1 = call_bifrost(
    "why people fall in Love and after some time hate the same person",
    GROQ_MODEL,
    fallback_model=GROQ_FALLBACK_MODEL,
)
print(content1)


provider=groq resolved_model=openai/gpt-oss-20b fallback_index=0 (primary) cache_hit=True hit_type=direct
fallbacks=['openai/gpt-4o-mini']
It’s a common, if painful, pattern: the spark that brings you together turns into frustration or disappointment over time. The “fall‑in‑love, fall‑in‑hate” cycle is usually a mix of psychological, emotional and situational factors. Below are some of the biggest pieces of the puzzle, plus a few suggestions on how to spot it early and, if you can, to keep the relationship healthy.

---

## 1. The “New‑Love” Over‑Excitation

### What’s happening?
- **Idealization** – In the first weeks or months, you focus on the positives and gloss over flaws. The brain releases dopamine and oxytocin, giving you a “high.”
- **Hyper‑attraction** – You’re often in a “honeymoon” mode, constantly noticing details that make you feel special.

### Why it can turn sour
- The brain’s reward system settles down, so the intensity fades.
- The real person, with all their idiosyn